In [1]:
import pandas as pd
import re
import logging
from pathlib import Path
from thefuzz import process, fuzz

# ==========================================
# 1. Configuration & Setup
# ==========================================
# Configure logging for better traceability
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Define file paths using pathlib for cross-platform compatibility
DATA_DIR = Path('./data/Assignment 1 - constraint_mapping')
OUTPUT_DIR = Path('./output')

# Create output directory if it doesn't exist
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ==========================================
# 2. Data Loading with Robust Separation
# ==========================================
def load_data(file_path: Path) -> pd.DataFrame:
    """
    Loads a CSV file into a pandas DataFrame, automatically trying common delimiters
    if a single column is detected.
    """
    try:
        # Step 1: Try reading with standard comma first
        df = pd.read_csv(file_path, sep=None, engine='python') 
        
        # If it still reads everything as 1 column, fallback to explicit tab
        if df.shape[1] == 1:
            df = pd.read_csv(file_path, sep='\t')
            
        logging.info(f"Loaded {file_path.name} -> Shape: {df.shape} | Columns found: {list(df.columns)}")
        
        # Clean column names (strip trailing/leading spaces that cause KeyErrors)
        df.columns = df.columns.str.strip()
        
        return df.fillna('')
    except FileNotFoundError:
        logging.error(f"File not found: {file_path}.")
        raise
    except Exception as e:
        logging.error(f"Error loading {file_path.name}: {e}")
        raise

# Reload the files using the smart loader
df_market = load_data(DATA_DIR / 'Market PJMISO constraint list.csv')
df_pano = load_data(DATA_DIR / 'Pano PJMISO constraint list.csv')
df_dayzer = load_data(DATA_DIR / 'Dayzer PJMISO constraint list.csv')

# ==========================================
# 3. Domain-Informed Normalization Logic
# ==========================================
def clean_power_string(text: str) -> str:
    text = str(text).upper().strip()
    text = re.sub(r'\s+', ' ', text)
    return text

def extract_pano_target(facility_str: str) -> str:
    match = re.search(r'\((.*?)\)', str(facility_str))
    if match:
        return clean_power_string(match.group(1))
    return clean_power_string(facility_str)

logging.info("Applying normalization heuristics...")

# Dynamic check to prevent script crashes if columns are named slightly differently
market_col = 'CONSTRAINT' if 'CONSTRAINT' in df_market.columns else df_market.columns[0]
pano_col = 'Monitored Facility' if 'Monitored Facility' in df_pano.columns else df_pano.columns[0]
dayzer_col = 'NAME' if 'NAME' in df_dayzer.columns else df_dayzer.columns[1] # usually 2nd column based on your screenshot

logging.info(f"Mapping using columns -> Market: '{market_col}', Pano: '{pano_col}', Dayzer: '{dayzer_col}'")

df_pano['normalized_target'] = df_pano[pano_col].apply(extract_pano_target)
df_market['normalized_constraint'] = df_market[market_col].apply(clean_power_string)
df_dayzer['normalized_name'] = df_dayzer[dayzer_col].apply(clean_power_string)

# Update dictionary mappings with the resolved columns
pano_dict = dict(zip(df_pano[pano_col], df_pano['normalized_target']))
dayzer_dict = dict(zip(df_dayzer[dayzer_col], df_dayzer['normalized_name']))

# ==========================================
# 4. Fuzzy Matching Engine
# ==========================================
def find_best_match(query: str, choices_dict: dict, scorer=fuzz.token_set_ratio, threshold: int = 80):
    """
    Finds the best match for a query within a dictionary of choices.
    Uses token_set_ratio by default to handle out-of-order words.
    
    Returns:
        tuple: (Best_Original_Name, Match_Score) or (None, 0)
    """
    if not query:
        return None, 0
    
    # process.extractOne searches through the dict values and returns: (matched_value, score, matched_key)
    match = process.extractOne(query, choices_dict, scorer=scorer)
    
    if match and match[1] >= threshold:
        matched_score = match[1]
        matched_original_key = match[2]
        return matched_original_key, matched_score
        
    return None, 0

# ==========================================
# 5. Execution: Cross-Source Mapping
# ==========================================
logging.info("Starting fuzzy matching process. This may take a moment depending on data size...")

mapping_results = []

for idx, row in df_market.iterrows():
    mkt_constraint = row['CONSTRAINT']
    mkt_contingency = row['CONTINGENCY']
    query = row['normalized_constraint']
    
    # Match against Panorama (high threshold since we extracted the exact sub-string)
    pano_match_name, pano_score = find_best_match(
        query=query, 
        choices_dict=pano_dict, 
        scorer=fuzz.token_set_ratio, 
        threshold=85
    )
    
    # Match against Dayzer (slightly lower threshold, using partial_ratio for broader interface matches)
    dayzer_match_name, dayzer_score = find_best_match(
        query=query, 
        choices_dict=dayzer_dict, 
        scorer=fuzz.partial_ratio, 
        threshold=80
    )
    
    # Append to results list
    mapping_results.append({
        'Market_Constraint': mkt_constraint,
        'Market_Contingency': mkt_contingency,
        'Dayzer_Constraint': dayzer_match_name,
        'Dayzer_Match_Score': dayzer_score,
        'Pano_Constraint': pano_match_name,
        'Pano_Match_Score': pano_score
    })

# Convert results to DataFrame
df_results = pd.DataFrame(mapping_results)

# ==========================================
# 6. Export Results & Reporting
# ==========================================
output_file = OUTPUT_DIR / 'constraint_mapping_results.csv'
df_results.to_csv(output_file, index=False)

logging.info(f"Mapping complete! Successfully processed {len(df_results)} records.")
logging.info(f"Results saved to: {output_file.resolve()}")

# Display the first few rows for validation in Jupyter Notebook
display(df_results.head(10))

2026-05-17 04:00:07,063 - INFO - Loaded Market PJMISO constraint list.csv -> Shape: (5230, 6) | Columns found: ['CONSTRAINT', 'CONTINGENCY', 'TOZONE', 'REPORTEDNAME', 'CONSTRAINTID', 'CONTINGENCYID']
2026-05-17 04:00:07,271 - INFO - Loaded Pano PJMISO constraint list.csv -> Shape: (21963, 5) | Columns found: ['PID', 'Monitored Facility', 'Contingency Name', 'Earliest', 'Latest']
2026-05-17 04:00:07,345 - INFO - Loaded Dayzer PJMISO constraint list.csv -> Shape: (13813, 2) | Columns found: ['CID', 'NAME']
2026-05-17 04:00:07,350 - INFO - Applying normalization heuristics...
2026-05-17 04:00:07,353 - INFO - Mapping using columns -> Market: 'CONSTRAINT', Pano: 'Monitored Facility', Dayzer: 'NAME'
2026-05-17 04:00:07,572 - INFO - Starting fuzzy matching process. This may take a moment depending on data size...
2026-05-17 04:08:21,715 - INFO - Mapping complete! Successfully processed 5230 records.
2026-05-17 04:08:21,718 - INFO - Results saved to: C:\Users\27403\OneDrive\Desktop\Upenn\inter

,Market_Constraint,Market_Contingency,Dayzer_Constraint,Dayzer_Match_Score,Pano_Constraint,Pano_Match_Score
0,NOTTINGH 230 KV NOTTINGHM 2-3 SER DEV,L500.CONASTONE-PEACHBOTTOM.5012,NOTTINGH_230 KV_NOT 2TR:ACTUAL,80,NOTTINGHM 2-3 SER DEV A 230 KV,97
1,LENOX 115 KV LENOX-NMESHOPP NML 1090,L230.ETOWANDA-HILLSIDE.2002 [NYISO],LENOX_115 KV_LEN-NME:ACTUAL,82,LENOX-NMESHOPP NML 1090 B 115 KV,100
2,EASTON 69 KV EAS-EMU,ACTUAL,EASTON_69 KV_EAS-EMU:ACTUAL,100,EASTON 69KV - EMUNI 69KV (EASTON 69 KV EAS-EMU),100
3,SAYRECON230 KV SAY-SAY,ACTUAL,SAYRECON_230 KV_SAY-SAY:ACTUAL,95,NaN,0
4,MOUN UGI 230 KV MOUN UGI 2 XFORMER,230/66.MOUNTAIN.T1 (SCTNLZ),MOUN UGI_230 KV_1:ACTUAL,80,MOUN UGI 69KV - MOUN UGI 230KV (MOUN UGI 230 K...,100
5,GRACETON 230 KV GRACETON-SAFEHARB 2303,L500.CONASTONE-PEACHBOTTOM.5012,NaN,0,GRACETON-SAFEHARB 2303 A 230 KV,100
6,94 HAURD-11323 11323 B 138 KV,L345.NELSON-ELECTRICJCT.15502,NaN,0,94 HAURD 138KV - 11323 138KV (94 HAURD 138 KV ...,100
7,BERGEN 230 KV BER-HUD,ACTUAL,BERGEN_230 KV_BER-HUD:ACTUAL,100,BERGEN 230KV - HCS 230KV (BERGEN 230 KV BER-HUD),100
8,LINE 69 KV MONR AE-VINELAND 0711-1,L230.CUMBERLAND-ORCHARD.2314,SHERMAN_69 KV_SHE-VIN:L69.Monroe-Vineland.0711,82,MONR AE-VINELAND 0711-1 A 69 KV,97
9,GARDNERS 115 KV GARDNERS-TEXSEAST GAR-TEX,L115.MIDDLETOWNJCT-COLLINS.975 (SCTNLZ),NaN,0,TEXSEAST 115KV - GARDNERS 115KV (GARDNERS 115 ...,100


In [ ]:

print("--- Dayzer Match Score Distribution ---")
print(df_results['Dayzer_Match_Score'].describe())

print("\n--- Pano Match Score Distribution ---")
print(df_results['Pano_Match_Score'].describe())

dayzer_unmatched = (df_results['Dayzer_Match_Score'] == 0).sum() / len(df_results) * 100
pano_unmatched = (df_results['Pano_Match_Score'] == 0).sum() / len(df_results) * 100

print(f"\nDayzer Unmatched Rate: {dayzer_unmatched:.2f}%")
print(f"Pano Unmatched Rate: {pano_unmatched:.2f}%")

--- Dayzer Match Score Distribution ---
count    5230.000000
mean       70.162715
std        39.243317
min         0.000000
25%        80.000000
50%        89.000000
75%        96.000000
max       100.000000
Name: Dayzer_Match_Score, dtype: float64

--- Pano Match Score Distribution ---
count    5230.000000
mean       70.978203
std        42.668638
min         0.000000
25%         0.000000
50%        97.000000
75%       100.000000
max       100.000000
Name: Pano_Match_Score, dtype: float64

Dayzer Unmatched Rate: 23.40%
Pano Unmatched Rate: 26.29%
